In [1]:
import pandas as pd
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import pickle

In [2]:
def build_features(df):
    """ determine features and labels, picked 7 features (7/14)
    returns (X, y) with numeric and dummy columns """
    numeric_X = df[['Age', 'Person Income', 'Loan Amount', 'Loan interest Rate', 'Credit Score']]
    dummies_X = pd.get_dummies(df[['Home Onwership', 'Previous Loan']])
    X = pd.concat([numeric_X, dummies_X], axis=1)
    y = df['Loan Status']
    return X, y

def build_pipeline():
    """ create pipeline with StandardScaler + SVC to use for training the model """
    return Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', SVC(kernel='rbf', class_weight='balanced'))
    ])


def evaluate(pipeline, X_test, y_test):
    """ print accuracy, precision, recall, f1 and confusion matrix
    return label (y) prediction """
    y_pred = pipeline.predict(X_test)
    print(f'The accuracy score is: {accuracy_score(y_test, y_pred)}')
    print(f'The precision score is: {precision_score(y_test, y_pred)}')
    print(f'The recall score is: {recall_score(y_test, y_pred)}')
    print(f'The f1 score is: {f1_score(y_test, y_pred)}')
    print(confusion_matrix(y_test, y_pred))
    return y_pred

def save_model(pipeline, filepath):
    """ save fitted pipeline to a .pkl file so it can be reloaded without retraining """
    with open(filepath, 'wb') as f:
        pickle.dump(pipeline, f)

In [3]:
##### Main #####

# load data and assemble features
df = pd.read_csv('p_data.csv')
X, y = build_features(df)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# model/pipeline creation & fit
pipeline = build_pipeline()
pipeline.fit(X_train, y_train)

# evaloation and df saving
evaluate(pipeline, X_test, y_test)
save_model(pipeline, 'loan_svc_model.pkl')

The accuracy score is: 0.8562222222222222
The precision score is: 0.6176084099868594
The recall score is: 0.9353233830845771
The f1 score is: 0.7439651760981401
[[5826 1164]
 [ 130 1880]]


In [4]:
# self check for fitted model
with open('loan_svc_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

y_load = loaded_model.predict(X_test)
print(confusion_matrix(y_test, y_load))

[[5826 1164]
 [ 130 1880]]


In [5]:
"""
As I ran tests over the model I realised something odd,
when I raised the person's income the approval percentages lowered, so I checked correspondence between:
"""

df['Person Income'].corr(df['Loan Status'])

np.float64(-0.13580771683574278)

In [6]:
"""
which made me realize that the higher income, the lower is approval rate
but that's probably not the only variable that's effecting.
so I took the ratio between loan amount and income and checked its correlation with loan status
"""
df['income_ratio'] = df['Loan Amount'] / df['Person Income']
df['income_ratio'].corr(df['Loan Status'])

np.float64(0.38501148705441535)

In [7]:
"""
that's what I've been looking for!
here we can see that there's a much stronger connection between the two
where the loan is a bigger part of person's income, the chance for a loan approval is higher

next I wanted to check the average of the ratio percentage where the loans got approved
"""
df.groupby('Loan Status')['income_ratio'].mean()

Loan Status
0    0.121810
1    0.202532
Name: income_ratio, dtype: float64

here we can definitely infer that the higher ratio of the loan per income
will positively affect the chance of loan approval
the data here shows that approved loans average 20% of annual income,
while declined loans average 12%

so in the end, this data is for experimental use only,
in the real world those ratios will probably matter in an opposite behavior